In [1]:
import torch.nn as nn
from torchvision import transforms, datasets
import torch
from torch.utils.data import DataLoader
import torch.nn.functional as F

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RESOLUTION = 252
DATA_ROOT = "/Users/alex/Developpement/Internship/datasets/aqua20/data/aqua20"

In [2]:
class simpleCNN(nn.Module):

    def __init__(self, num_classes=10, depth=2, resolution=RESOLUTION):
        super(simpleCNN, self).__init__()
        layers = []
        channels = 3
        base_channels = [64 * (2 ** i) for i in range(depth)]
        for i in range(depth):
            layers.append(nn.Conv2d(channels, base_channels[i], kernel_size=3, padding=1))
            layers.append(nn.ReLU())
            layers.append(nn.MaxPool2d(kernel_size=2, stride=2))
            channels = base_channels[i]

        final_spatial = resolution
        for _ in range(depth):
            final_spatial //= 2
        layers.append(nn.Flatten())
        layers.append(nn.Linear(channels * final_spatial ** 2, num_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

In [ ]:
from torch.optim import Optimizer
from torch.nn import Module
from torch.nn.modules.loss import _Loss
from tqdm.notebook import tqdm 

def train_epoch(model: Module, loader: DataLoader, optimizer: Optimizer, lossfn: _Loss, device):
    model.train()
    training_loss = 0.0
    training_acc = 0
    total = 0
    for x,y in tqdm(loader, desc="train", leave=False):
        x,y = x.to(device), y.to(device)
        optimizer.zero_grad()
        outputs = model(x)
        loss = lossfn(outputs, y)
        loss.backward()
        optimizer.step()
        training_loss += loss.item() * x.size(0)
        preds = outputs.argmax(dim=1)
        training_acc += (preds==y).sum().item()
        total += x.size(0)

    return training_loss/total, training_acc/total

In [4]:
def evaluate(model: Module, loader: DataLoader, lossfn: _Loss, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for x,y in loader:
            x,y = x.to(device), y.to(device)
            outputs = model(x)
            loss = lossfn(outputs, y)
            running_loss += loss.item()
            preds = outputs.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += x.size(0)

    avg_eval_loss = running_loss/total
    acc = correct/total
    return avg_eval_loss, acc

In [5]:
transform = transforms.Compose([
    transforms.Resize(RESOLUTION),
    transforms.CenterCrop(RESOLUTION),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

In [6]:
test_ds    = datasets.ImageFolder(f"{DATA_ROOT}/test",  transform=transform)
test_loader = DataLoader(test_ds,   batch_size=64, shuffle=False, num_workers=4)

## Full data

In [7]:
full_train = datasets.ImageFolder(f"{DATA_ROOT}/train", transform=transform)
full_loader = DataLoader(full_train, batch_size=64, shuffle=True, num_workers=4)

In [8]:
cnn = simpleCNN(num_classes=20, depth=3).to(DEVICE)
optimizer = torch.optim.Adam(cnn.parameters(), lr=1e-3)
lossfn = nn.CrossEntropyLoss()

In [12]:
print("Training...")
history_cnn = {'train_loss':[], 'train_acc':[], 'test_loss':[], 'test_acc':[]}

for epoch in range(10):
    train_loss, train_acc = train_epoch(cnn, full_loader, optimizer, lossfn, DEVICE)
    test_loss, test_acc = evaluate(cnn, test_loader, lossfn, DEVICE)
    history_cnn['train_loss'].append(train_loss)
    history_cnn['train_acc'].append(train_acc)
    history_cnn['test_loss'].append(test_loss)
    history_cnn['test_acc'].append(test_acc)
    print(f"Epoch {epoch+1}/10 - Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}")

Training...


Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/Users/alex/.local/share/uv/python/cpython-3.12.12-macos-aarch64-none/lib/python3.12/multiprocessing/spawn.py", line 122, in spawn_main

    exitcode = _main(fd, parent_sentinel)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/alex/.local/share/uv/python/cpython-3.12.12-macos-aarch64-none/lib/python3.12/multiprocessing/spawn.py", line 132, in _main
    self = reduction.pickle.load(from_parent)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/alex/Developpement/Internship/GradientDistillation/.venv/lib/python3.12/site-packages/torchvision/__init__.py", line 8, in <module>
    from torchvision import _meta_registrations, datasets, io, models, ops, transforms, utils  # usort:skip
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/alex/Developpement/Internship/GradientDistillation/.venv/lib/python3.12/site-packages/torchvision/data

KeyboardInterrupt: 

### Distilled data

In [ ]:
DISTILLED_PTH = "../logged_files/distillation/aqua20/dinov2_vitb/dinov2_vitb_distill_252_ipc1_augs10_physics/data.pth"
distilled = torch.load(DISTILLED_PTH, map_location=DEVICE)
# distilled est un tensor (N, C, H, W) ou un dict selon ton format
# adapte selon ce que run.sh sauvegarde
print(f"Distilled data keys: {distilled.keys()}")
images_d = distilled["images"].to(DEVICE)   # shape: (20, 3, 196, 196)
labels_d = distilled["labels"].to(DEVICE)
print(labels_d)
from torch.utils.data import TensorDataset
distill_loader = DataLoader(
    TensorDataset(images_d.cpu(), labels_d.cpu()),
    batch_size=20, shuffle=True
)

Distilled data keys: dict_keys(['syn_J', 'syn_T', 'syn_B', 'images', 'labels'])
tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19])


In [ ]:
cnn_distill = simpleCNN(num_classes=20, depth=3).to(DEVICE)
optimizer = torch.optim.Adam(cnn_distill.parameters(), lr=1e-3)
lossfn = nn.CrossEntropyLoss()

In [ ]:
print("Training...")
history_cnn_distill = {'train_loss':[], 'train_acc':[], 'test_loss':[], 'test_acc':[]}

for epoch in range(10):
    train_loss, train_acc = train_epoch(cnn_distill, distill_loader, optimizer, lossfn, DEVICE)
    test_loss, test_acc = evaluate(cnn_distill, test_loader, lossfn, DEVICE)
    history_cnn_distill['train_loss'].append(train_loss)
    history_cnn_distill['train_acc'].append(train_acc)
    history_cnn_distill['test_loss'].append(test_loss)
    history_cnn_distill['test_acc'].append(test_acc)
    print(f"Epoch {epoch+1}/10 - Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}")

Training...
Epoch 1/10 - Train Loss: 2.9960, Train Acc: 0.0500, Test Loss: 0.0792, Test Acc: 0.0242
Epoch 2/10 - Train Loss: 3.0603, Train Acc: 0.0500, Test Loss: 0.1537, Test Acc: 0.0217


KeyboardInterrupt: 